# Rosetta — a short demo

**What this notebook shows:**

1. What's in the Rosetta catalog (the data sources it can serve).
2. How to check that a source is healthy before fetching.
3. Fetching a **seasonal forecast** (ECMWF via C3S) for a configurable region and init date.
4. Fetching **observations** (CHIRPS precipitation) for the same region.
5. **Unit harmonization** — each provider comes back in Rosetta's units for that temporal contract.
6. **Exporting to GeoTIFF** for GIS handoff.
7. Verifying the GeoTIFFs render correctly on a basemap.
8. **Storage surfaces** — in-memory, local cache, NetCDF / GeoTIFF on disk, and **direct write to S3 buckets** for downstream pipelines.

All region and date knobs are in the **Configuration** cell — change them once at the top and the rest of the notebook follows.

---

**Prerequisites**

- `uv sync` has been run at the Rosetta repo root.
- Demo extras installed: `uv pip install 'lab/rosetta[demo]'` (adds matplotlib + cartopy + rasterio).
- For C3S / ERA5 products: CDS credentials at `~/.cdsapirc` (see top-level README).
- CHIRPS, ERSST, NMME S3 sources need no auth.
- S3 writes (optional) require AWS credentials resolvable by `s3fs` (env vars, `~/.aws/credentials`, or an IAM role).


## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import rosetta

OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Rosetta loaded. Outputs will be written to {OUTPUT_DIR.resolve()}")


## Configuration

Change these values to re-run the notebook for a different region or forecast.

- `REGION = [lat_s, lat_n, lon_w, lon_e]` in degrees (lon in -180..180).
- `INIT` is the forecast initialization month as `YYYY-MM`.
- `TARGET_SEASON` is a three-letter season (DJF, JFM, FMA, MAM, ...).
- `OBS_YEAR_RANGE` is `(start_year, end_year)` for the observations fetch.

In [ ]:
REGION = [5, 20, -20, 20]         # West Africa / Sahel — [lat_s, lat_n, lon_w, lon_e]
INIT = "2024-02"                  # forecast init: February 2024
TARGET_SEASON = "MAM"             # predict March-April-May
OBS_YEAR_RANGE = (2022, 2023)     # CHIRPS window

FORECAST_PRODUCT = "c3s/ecmwf"
OBS_PRODUCT = "obs/chirps"
VARIABLE = "precip"

# cartopy wants [lon_w, lon_e, lat_s, lat_n]
EXTENT = [REGION[2], REGION[3], REGION[0], REGION[1]]

print(f"Region (lat_s, lat_n, lon_w, lon_e): {REGION}")
print(f"Cartopy extent (lon_w, lon_e, lat_s, lat_n): {EXTENT}")
print(f"Forecast: {FORECAST_PRODUCT}, init={INIT}, target={TARGET_SEASON}")
print(f"Observations: {OBS_PRODUCT}, years={OBS_YEAR_RANGE}")


## 1. What's in the catalog?

Rosetta is a **catalog of adapters**. Each entry is a provider + variable mapping that Rosetta knows how to fetch and normalize. The catalog is a YAML file — sources are added by editing it, not by code changes.

In [ ]:
products = rosetta.catalog.list_products()

groups = {}
for p in products:
    prefix = p.split("/")[0]
    groups.setdefault(prefix, []).append(p)

for prefix, entries in sorted(groups.items()):
    print(f"\n{prefix}/  ({len(entries)} products)")
    for e in entries:
        print(f"  - {e}")

Each catalog entry has structured metadata — adapter type, upstream URL, variable list, native units, and the conversion to Rosetta's canonical units:

In [ ]:
import json
print(json.dumps(rosetta.catalog.get(FORECAST_PRODUCT), indent=2, default=str))

## 2. Is the source healthy?

Before fetching, ask Rosetta whether the upstream provider looks reachable and the catalog entry parses correctly. This is the mechanism that a future operational dashboard would poll on a schedule.

In [ ]:
status = rosetta.check_product(FORECAST_PRODUCT, probe_remote=True)
print(json.dumps(status, indent=2, default=str))

## 3. Fetch a seasonal forecast

`rosetta.fetch()` is the one call you need. It:

- selects the right adapter for the product,
- downloads only the region and lead times required,
- normalizes coordinates (`lat`, `lon`, `init_time`, `lead_time`, `member`),
- converts units to Rosetta's canonical ones (this lead-resolved C3S forecast is `mm/day`; collapsed seasonal output with `year_index=True` is `mm` across providers),
- returns a ready-to-use `xarray.Dataset`.

This cell hits CDS. First-time fetches can take several minutes while the CDS queue processes the request; subsequent calls with identical arguments come from Rosetta's local cache (`~/.cache/rosetta/`).

In [ ]:
forecast = rosetta.fetch(
    product=FORECAST_PRODUCT,
    variable=VARIABLE,
    init=INIT,
    target=TARGET_SEASON,
    region=REGION,
)
forecast

Note the dims: `(member, lead_time, lat, lon)`. No matter which forecast provider you pull from, the shape is the same — that's Rosetta's job.

Quick ensemble-mean map of the first lead month:

## Plotting helpers

Two small utilities used by the plots below — kept here so the flow stays visible. `summary()` prints min/max/NaN so empty maps are immediately diagnosable; `plot_on_basemap()` puts a 2D DataArray on a cartopy basemap with coastlines and country borders.


In [ ]:
def summary(da, label):
    v = da.values
    finite = np.isfinite(v)
    n_tot = v.size
    n_nan = int((~finite).sum())
    if not finite.any():
        print(f"{label}: ALL NaN ({n_nan}/{n_tot}) — plot will be blank")
        return
    vf = v[finite]
    print(f"{label}: min={vf.min():.3g}, max={vf.max():.3g}, mean={vf.mean():.3g}, "
          f"NaN={n_nan}/{n_tot} ({100 * n_nan / n_tot:.1f}%)")


def plot_on_basemap(da, title, extent, units=""):
    """Plot a 2D DataArray on a PlateCarree basemap. extent = [lon_w, lon_e, lat_s, lat_n]."""
    plt.figure(figsize=(9, 5))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    da.plot(ax=ax, transform=ccrs.PlateCarree(),
            cbar_kwargs={"label": units, "shrink": 0.7}, add_labels=False)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="gray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
ensmean = forecast[VARIABLE].mean(dim="member")
first_lead = ensmean.isel(lead_time=0)

summary(first_lead, f"forecast ens-mean, lead 1")
plot_on_basemap(
    first_lead,
    title=f"{FORECAST_PRODUCT}: {VARIABLE}, init={INIT}, lead 1 — ensemble mean",
    extent=EXTENT,
    units=forecast[VARIABLE].attrs.get("units", ""),
)


## 4. Fetch observations

Same API, different product. CHIRPS precipitation over the same region:

In [ ]:
obs = rosetta.fetch(
    product=OBS_PRODUCT,
    variable=VARIABLE,
    region=REGION,
    hindcast=OBS_YEAR_RANGE,
)
obs

In [ ]:
obs_mean = obs[VARIABLE].mean(dim="time")

summary(obs_mean, f"obs mean over {OBS_YEAR_RANGE[0]}–{OBS_YEAR_RANGE[1]}")
plot_on_basemap(
    obs_mean,
    title=f"{OBS_PRODUCT}: {VARIABLE}, mean over {OBS_YEAR_RANGE[0]}–{OBS_YEAR_RANGE[1]}",
    extent=EXTENT,
    units=obs[VARIABLE].attrs.get("units", ""),
)


## 5. Units follow a documented temporal contract

Providers ship the same variable in different native units. Rosetta normalizes those units while preserving the requested temporal meaning: lead-resolved or daily precipitation is generally `mm/day`, while collapsed seasonal forecasts (`year_index=True` or `assemble()`) are `mm` across NMME, C3S/CDS, and IRI. Temperatures are Kelvin in some GCMs and Celsius in observations. Always use the returned variable's `units` attribute when labeling output.

The translation rules are declared per-variable in the catalog, so they're auditable at a glance:

In [ ]:
print(f"{'product':28} {'variable':8} {'native':22} → canonical")
print("-" * 72)
for p in sorted(rosetta.catalog.list_products()):
    for var, v in rosetta.catalog.get(p).get("variables", {}).items():
        print(f"{p:28} {var:8} {v.get('units','?'):22} → {v.get('target_units','?')}")

And here's the proof on the data we just fetched — two different providers, two different native units, identical canonical output:

In [ ]:
for label, ds, product in [("forecast",     forecast, FORECAST_PRODUCT),
                           ("observations", obs,      OBS_PRODUCT)]:
    native = rosetta.catalog.get(product)["variables"][VARIABLE].get("units", "?")
    output = ds[VARIABLE].attrs.get("units", "?")
    mean   = float(ds[VARIABLE].mean())
    print(f"{label:13} ({product:14})  native={native:22}  output={output:8}  mean={mean:.3g}")

## 6. Export to GeoTIFF

This is the primary integration path for a Data Library. `rosetta.fetch(..., destination=..., format="geotiff")` writes the result as a GeoTIFF in EPSG:4326, with one band per lead time (forecasts) or per time step (observations).

For forecasts, `member` is reduced to an ensemble mean before writing — GeoTIFF is a 2D/multi-band format, so something has to give. If you want per-member output, loop and write one TIFF per member.

We'll re-run the fetch with a `destination` and `format` set. The cache means the network isn't touched again.

In [ ]:
forecast_tif = OUTPUT_DIR / f"{FORECAST_PRODUCT.replace('/', '_')}_{VARIABLE}_{INIT}_{TARGET_SEASON}.tif"

rosetta.fetch(
    product=FORECAST_PRODUCT,
    variable=VARIABLE,
    init=INIT,
    target=TARGET_SEASON,
    region=REGION,
    destination=str(forecast_tif),
    format="geotiff",
)
print(f"Wrote {forecast_tif}")

In [ ]:
obs_tif = OUTPUT_DIR / f"{OBS_PRODUCT.replace('/', '_')}_{VARIABLE}_{OBS_YEAR_RANGE[0]}_{OBS_YEAR_RANGE[1]}.tif"

rosetta.fetch(
    product=OBS_PRODUCT,
    variable=VARIABLE,
    region=REGION,
    hindcast=OBS_YEAR_RANGE,
    destination=str(obs_tif),
    format="geotiff",
)
print(f"Wrote {obs_tif}")

## 7. Inspect the GeoTIFFs

The files are standard GeoTIFFs — they open in QGIS, ArcGIS, `gdalinfo`, `rasterio`, or anything else that reads TIFF. Here's a quick `rasterio` peek:

In [ ]:
for path in [forecast_tif, obs_tif]:
    with rasterio.open(path) as src:
        print(f"\n{path.name}")
        print(f"  CRS:          {src.crs}")
        print(f"  bands:        {src.count}")
        print(f"  shape (h,w):  {src.shape}")
        print(f"  bounds:       {src.bounds}")
        print(f"  descriptions: {src.descriptions}")

## 8. Verify the GeoTIFFs on a basemap

The real integration test: load the exported `.tif` files with `rioxarray` (no Rosetta involved — this is what any downstream consumer would do), then overlay them on coastlines and country borders. If the precipitation patterns sit over the ITCZ where they should, the round-trip is working.


In [ ]:
import rioxarray

for tif_path in [forecast_tif, obs_tif]:
    da = rioxarray.open_rasterio(tif_path).rename({"x": "lon", "y": "lat"})
    # Pick band 1 (first lead month for forecasts; first time step for obs)
    band1 = da.isel(band=0)
    with rasterio.open(tif_path) as src:
        band_label = src.descriptions[0] or "band 1"

    summary(band1, f"{tif_path.name} band 1")
    plot_on_basemap(
        band1,
        title=f"{tif_path.name} — {band_label}",
        extent=EXTENT,
        units=band1.attrs.get("units", ""),
    )


## 9. Storage surfaces Rosetta gives you today

Rosetta is deliberately **not** a data warehouse — raw data stays at the provider, and Rosetta's job is the translation layer. That said, every `fetch()` call produces a normalized result that can land in any of the surfaces below. A Data Library (or an ops pipeline) can pick whichever fits its handoff.

| # | Surface | Call shape | Role |
|---|---|---|---|
| 1 | **In-memory xarray** | `ds = rosetta.fetch(...)` — the default return | Interactive work. Nothing is written. Lives until the kernel restarts. |
| 2 | **Local disk cache** (transparent) | automatic — written to `~/.cache/rosetta/` keyed by the full request | Second fetch with the same arguments returns instantly, no network. Toggle with `rosetta.set_cache(enabled=False)`; bust by deleting the directory. |
| 3 | **NetCDF — local** | `rosetta.fetch(..., destination="x.nc")` | The canonical scientific format — CF-compliant, opens in Python, R, MATLAB, CDO, Panoply. Preserves every dim (`member`, `lead_time`, `init_time`). |
| 4 | **NetCDF — S3 bucket** | `rosetta.fetch(..., destination="s3://bucket/key.nc")` | Same NetCDF, written straight to object storage via `s3fs`. The primary handoff for a cloud Data Library: one call deposits a fully-normalized file at a stable URL that downstream systems can pull from. |
| 5 | **GeoTIFF — local** | `rosetta.fetch(..., destination="x.tif", format="geotiff")` | GIS-native handoff. EPSG:4326, north-up, multi-band per lead/time. Demoed in §6–8. (GeoTIFF → S3 is not yet wired — use local + a separate upload step for now.) |

The cell below touches each surface so the set is visible in one place. The S3 write is shown as an inert call-shape example — uncomment and set `S3_DEST` to run it against a bucket you own.


In [ ]:
# 1. In-memory: already happened — `forecast` and `obs` are live xarray Datasets.
print("(1) in-memory:", type(forecast).__name__, "with dims", dict(forecast.sizes))

# 2. Cache: show the cache directory + its contents after our earlier fetches.
from pathlib import Path
cache_dir = Path.home() / ".cache" / "rosetta"
cached = sorted(cache_dir.glob("*.nc")) if cache_dir.exists() else []
print(f"(2) cache:     {cache_dir}  ({len(cached)} entr{'y' if len(cached)==1 else 'ies'})")
for p in cached[:3]:
    print(f"                  {p.name}  ({p.stat().st_size/1e6:.1f} MB)")

# 3. NetCDF (local): save the obs we already have in memory. Cache hit, no re-fetch.
obs_nc = OUTPUT_DIR / f"{OBS_PRODUCT.replace('/', '_')}_{VARIABLE}_{OBS_YEAR_RANGE[0]}_{OBS_YEAR_RANGE[1]}.nc"
rosetta.fetch(
    product=OBS_PRODUCT, variable=VARIABLE, region=REGION,
    hindcast=OBS_YEAR_RANGE,
    destination=str(obs_nc), format="netcdf",
)
print(f"(3) NetCDF:    {obs_nc.name}  ({obs_nc.stat().st_size/1e6:.1f} MB)")

# 4. NetCDF -> S3 bucket. Call shape shown; uncomment and point at a real bucket to run.
# S3_DEST = "s3://your-bucket/rosetta-demo/chirps_precip.nc"
# rosetta.fetch(
#     product=OBS_PRODUCT, variable=VARIABLE, region=REGION,
#     hindcast=OBS_YEAR_RANGE,
#     destination=S3_DEST, format="netcdf",
# )
print("(4) S3 NetCDF: call-shape only — set S3_DEST and uncomment above to run.")

# 5. GeoTIFF: already produced above.
print(f"(5) GeoTIFF:   {obs_tif.name}  ({obs_tif.stat().st_size/1e6:.1f} MB)")


## Where to go from here

- **Try a different provider.** Change `FORECAST_PRODUCT` to `c3s/ukmo`, `c3s/meteofrance`, `nmme/cfsv2`, etc. — the rest of the notebook doesn't change.
- **Different variable.** `tmax`, `tmin`, `sst` (depending on the product's `variables` list — check `rosetta.catalog.get(product)`).
- **S3 handoff.** Flip any `destination=` to an `s3://…` URL to write straight to a bucket (NetCDF). This is the path a cloud Data Library would actually use.
- **Parity validation.** Rosetta ships a separate harness (`scripts/validate_against_iri.py`) that runs the full IRIDL comparison offline — that's where the numerical parity check lives, rather than in this demo.
- **Operational deployment.** `check_all_products()` returns a health-status list for every product in the catalog; a future operational dashboard would poll this on a schedule and surface red/yellow/green per source.